In [ ]:
import kagglehub
import os
import pandas as pd
import matplotlib.pyplot as plt
import json
import networkx as nx
from difflib import get_close_matches

c:\Users\dav-9\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Liste actor

In [2]:
# Download latest version
path = kagglehub.dataset_download("tmdb/tmdb-movie-metadata")

#print("Path to dataset files:", path)
#print("Fichiers dans le dossier :")
#print(os.listdir(path))

file_path_movie = os.path.join(path, "tmdb_5000_movies.csv")
file_path_credits = os.path.join(path, "tmdb_5000_credits.csv")

movies   = pd.read_csv(file_path_movie)
credits  = pd.read_csv(file_path_credits)

# Vérifier les colonnes
#print(credits.columns)
#print(movies.columns)

# Faire la jointure sur movie_id (credits) et id (movies)
data = movies.merge(credits, left_on='id', right_on='movie_id', how='inner')

"""print(data.shape)
col = data.columns.tolist()
for i in range(0, len(col), 5) :
    if (i + 5 <= len(col)) :
        print(col[i : i + 5])
    else :
        print(col[i:])
print(data.head())"""

'print(data.shape)\ncol = data.columns.tolist()\nfor i in range(0, len(col), 5) :\n    if (i + 5 <= len(col)) :\n        print(col[i : i + 5])\n    else :\n        print(col[i:])\nprint(data.head())'

In [3]:
def get_top_actors(cast_json, nbr) :
    try :
        cast_list = json.loads(cast_json)
        top = [actor['name'] for actor in cast_list]
        while len(top) < nbr :
            top.append(None)
        return top[:nbr]
    except :
        return [None] * nbr

def get_imdb_graph(data, nbr_actors=3, firstyear=1, lastyear=3000, genre=None, minscore=0, maxscore=10) :

    df = data.copy()

    # Filtre par genre
    if genre is not None:
        df = df[df['genres'].str.contains(genre, na=False)]

    # S'assurer que release_date est bien en datetime
    df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
    # Extraire l'année
    df['release_year'] = df['release_date'].dt.year
    # Filtre par année et score
    df = df[(df['release_year'] >= firstyear) &
            (df['release_year'] <= lastyear) &
            (df['vote_average'] >= minscore) &
            (df['vote_average'] <= maxscore)]

    # Sélection des colonnes utiles
    df = df[['original_title', 'cast', 'release_year']]

    # Garder seulement les lignes où 'cast' contient au moins nbr_actors acteurs
    #df = df[df['cast'].apply(lambda x: len(json.loads(x)) >= nbr_actors)]

    # Une seule colonne 'actors' contenant une liste
    df['actors'] = df['cast'].apply(lambda x: get_top_actors(x, nbr_actors))

    # Création du graphe
    G = nx.Graph()

    for _, row in df.iterrows() :
        # Retirer les valeurs None de la liste
        actors = [a for a in row['actors'] if pd.notna(a)]

        # Ajouter toutes les combinaisons d'acteurs
        if len(actors) >= 2 :
            for i in range(len(actors)) :
                for j in range(i + 1, len(actors)) :
                    G.add_edge(actors[i], actors[j])

    return G

In [4]:
G = get_imdb_graph(data, nbr_actors=10, firstyear=1990, lastyear=2009)

print("\tNombre d'acteurs :", G.number_of_nodes())
print("\tNombre de collaborations :", G.number_of_edges())

	Nombre d'acteurs : 11637
	Nombre de collaborations : 111299


# MBTI

In [5]:
path_mbti = kagglehub.dataset_download("yuraslastya/celeb-mbti")

print("Path to dataset files :", path_mbti)
print("Fichiers dans le dossier :")
print(os.listdir(path_mbti))

Path to dataset files : C:\Users\dav-9\.cache\kagglehub\datasets\yuraslastya\celeb-mbti\versions\1
Fichiers dans le dossier :
['mbti_celebrities.csv']


In [ ]:
file_path_mbti = os.path.join(path_mbti, "mbti_celebrities.csv")

mbti   = pd.read_csv(file_path_mbti)

print(mbti.shape)
# Vérifier les colonnes
print("=== Colonnes ===")
print(mbti.columns.tolist())

print("\n=== Aperçu ===")
print(mbti.head(10).to_string())



(50878, 32)
=== Colonnes ===
['id', 'name', 'cat_id', 'category', 'sub_cat_id', 'subcategory', 'four_letter', 'four_letter_total_voted', 'enneagram', 'enneagram_total_voted', 'socionics', 'socionics_total_voted', 'instinctual_variant', 'instinctual_variant_total_voted', 'tritype', 'tritype_total_voted', 'temperaments', 'temperaments_total_voted', 'attitudinal_psyche', 'attitudinal_psyche_total_voted', 'big_5_SLOAN', 'big_5_SLOAN_total_voted', 'classic_jungian', 'classic_jungian_total_voted', 'letter_1', 'letter_1_percentage', 'letter_2', 'letter_2_percentage', 'letter_3', 'letter_3_percentage', 'letter_4', 'letter_4_percentage']

=== Aperçu ===
      id                    name  cat_id category  sub_cat_id subcategory four_letter  four_letter_total_voted enneagram  enneagram_total_voted socionics  socionics_total_voted instinctual_variant  instinctual_variant_total_voted  tritype  tritype_total_voted           temperaments  temperaments_total_voted attitudinal_psyche  attitudinal_psyche

In [38]:
print(len(mbti['subcategory'].value_counts()))
print()
for name, count in mbti['subcategory'].value_counts().items():
    print(f"{name} : {count}")

184

General Vloggers : 2911
European, Musicians : 2472
Actors and Actresses (USA) : 2427
Internet Personalities (Other) : 1813
Writers (Literature, Modern) : 1765
Comedians : 1714
Kpop : 1689
Actors & Actresses (Asia) : 1656
Criminals : 1275
Football (Soccer) : 1220
TikTok Stars : 1199
Historical Figures (1900s) : 1008
Voice Acting : 991
Football (American) : 928
Hosts & Presenters : 912
Actors & Actresses (Europe) : 822
Online Fictional Characters : 821
Actors and Actresses (UK & Ireland) : 801
Brazil, Musicians : 720
Film Directors : 716
Writers (Literature, Classic) : 709
News & Journalists : 693
Baseball : 685
Virtual Youtubers : 681
Artists : 658
Business : 646
Artists & Animators : 617
Latin American, Musicians : 536
Actors & Actresses (Latin America) : 502
Government (USA) : 486
Models : 485
Scientists, Technology & Educators : 477
Historical Figures (1800s) : 446
Wrestling : 443
Hosts, Critics, Producers & Editors : 436
Psychology & Personal Development : 430
Performers : 409


### Some informations

In [12]:
# Liste d'acteurs à rechercher
actors_to_find = ["Frankie Muniz", "Samuel L. Jackson"]

result = mbti[mbti['name'].isin(actors_to_find)]
print(result.to_string())

         id               name  cat_id     category  sub_cat_id                 subcategory four_letter  four_letter_total_voted enneagram  enneagram_total_voted socionics  socionics_total_voted instinctual_variant  instinctual_variant_total_voted  tritype  tritype_total_voted          temperaments  temperaments_total_voted attitudinal_psyche  attitudinal_psyche_total_voted big_5_SLOAN  big_5_SLOAN_total_voted classic_jungian  classic_jungian_total_voted letter_1  letter_1_percentage letter_2  letter_2_percentage letter_3  letter_3_percentage letter_4  letter_4_percentage
820     610  Samuel L. Jackson       1  Pop Culture          13  Actors and Actresses (USA)        ESTP                      226       8w7                     97       SEE                     22               so/sp                               26    873.0                   21   Sanguine [Dominant]                        24               FVLE                               5       SCUEN                       16        

### DB MBTI with actors only

In [10]:
# Filtrer uniquement les acteurs (category == 'Pop Culture' et subcategory contenant 'Actor')
actors_mbti = mbti[mbti['subcategory'].str.contains('Actor', na=False)]

print(f"Nombre d'acteurs dans la db MBTI : {len(actors_mbti):,}")
print(actors_mbti['subcategory'].value_counts())

Nombre d'acteurs dans la db MBTI : 6,915
subcategory
Actors and Actresses (USA)             2427
Actors & Actresses (Asia)              1656
Actors & Actresses (Europe)             822
Actors and Actresses (UK & Ireland)     801
Actors & Actresses (Latin America)      502
Actors & Actresses (World)              341
Actors & Actresses (Canada)             223
Actors & Actresses (Oceania)            143
Name: count, dtype: int64


### Perfect match

In [25]:
# Récupérer les acteurs du graphe
actors_in_graph = set(G.nodes())

# Récupérer les acteurs dans la db MBTI
actors_in_mbti = set(mbti['name'].tolist())

# Intersection : acteurs présents dans les deux
common_actors = actors_in_graph & actors_in_mbti

print(f"Acteurs dans le graphe         : {len(actors_in_graph):,}")
print(f"Célébrités dans MBTI           : {len(actors_in_mbti):,}")
print(f"Acteurs en commun              : {len(common_actors):,}")

Acteurs dans le graphe         : 11,637
Célébrités dans MBTI           : 50,640
Acteurs en commun              : 2,401


In [27]:
# Récupérer les acteurs du graphe
actors_in_graph = set(G.nodes())

# Récupérer les acteurs dans la db MBTI
actors_in_mbti = set(actors_mbti['name'].tolist())

# Intersection : acteurs présents dans les deux
common_actors_2 = actors_in_graph & actors_in_mbti

print(f"Acteurs dans le graphe         : {len(actors_in_graph):,}")
print(f"Célébrités dans MBTI           : {len(actors_in_mbti):,}")
print(f"Acteurs en commun              : {len(common_actors_2):,}")

Acteurs dans le graphe         : 11,637
Célébrités dans MBTI           : 6,912
Acteurs en commun              : 1,912


In [39]:
difference = common_actors - common_actors_2
actors_to_find = []
print(len(difference))
print()
for i in list(difference) :
    actors_to_find.append(i)

result = mbti[mbti['name'].isin(actors_to_find)]
print(len(result['subcategory'].value_counts()))
print()
for name, count in result['subcategory'].value_counts().items():
    print(f"{name} : {count}")
#print(result[:10].to_string())

489

62

Comedians : 94
People of Classic Hollywood : 63
Voice Acting : 55
Film Directors : 49
Hosts & Presenters : 25
Models : 19
Writers (Literature, Modern) : 13
Basketball : 11
Film & TV Crew : 9
News & Journalists : 8
Classic Pop & Contemporary : 8
Football (American) : 8
Government (USA) : 7
Criminals : 6
Artists (Animators) : 6
Wrestling : 6
European, Musicians : 6
Martial Arts : 5
Baseball : 5
Business : 4
Historical Figures (1900s) : 4
Latin American, Musicians : 4
MMA : 4
Boxing : 4
Presidents of the USA : 4
Artists : 3
Football (Soccer) : 3
Famous For Being Famous : 3
Political Commentators : 3
Performers : 3
Internet Personalities (Other) : 3
General Vloggers : 3
Skateboarding : 3
Brazil, Musicians : 3
Physics & Astronomy : 3
Writers (Comics) : 2
Wrestling (The Performers) : 2
Activists : 2
Skiing & Snowboarding : 2
Tennis : 2
Mathematics : 2
Psychology & Neuroscience : 2
Classical : 2
Kpop : 2
Hockey : 1
Government (World) : 1
Culinary Arts : 1
People of Law : 1
Ice skatin

### Match even if not perfect

In [40]:
# Récupérer les acteurs du graphe
actors_in_graph = set(G.nodes())
# Récupérer les acteurs dans la db MBTI
actors_in_mbti = set(mbti['name'].tolist())
# Intersection : acteurs présents dans les deux
common_actors = actors_in_graph & actors_in_mbti

# Pour chaque acteur du graphe, chercher le nom le plus proche dans MBTI
mbti_names = set(mbti['name'].tolist())
mbti_names = list(mbti_names - common_actors) # on enleve les perfect match
actors_in_graph = actors_in_graph - common_actors # on enleve les perfect match 
matched = {}

for actor in actors_in_graph :
    matches = get_close_matches(actor, mbti_names, n=1, cutoff=0.9)
    if matches :
        matched[actor] = matches[0]

print(f"Acteurs matchés (approx)  : {len(matched):,}")
print("\nNoms match :")
for original, match in list(matched.items()) :
    if original != match :
        print(f"  {original:<30} match '{match}'")

Acteurs matchés (approx)  : 223

Noms match :
  Mark Walton                    match 'Mark Watson'
  Donny Gray                     match 'Sonny Gray'
  George Perez                   match 'George Pérez'
  Pierre Curzi                   match 'Pierre Curie'
  Rob Moran                      match 'Rob Morgan'
  Konstantin Khabenskiy          match 'Konstantin Khabensky'
  Paul Sorvino                   match 'Paul Soriano'
  Sue Johnston                   match 'Sue Johanson'
  Sam Behrens                    match 'Sam Berns'
  Jordan Garrett                 match 'Jordan Barrett'
  Daniel Polo                    match 'Daniel Molo'
  Sam Huntington                 match 'Samuel Huntington'
  Michael Bunin                  match 'Michael Bunting'
  Robert Hoffman                 match 'Robert Hoffmann'
  Murilo Benicio                 match 'Murilo Benício'
  Max Martini                    match 'Max Martin'
  Mark Mason                     match 'Mark Manson'
  Moisés Arias           

In [41]:
# Créer un dict name -> mbti
mbti_dict = mbti.set_index('name')['four_letter'].to_dict()

# Ajouter le MBTI comme attribut de chaque nœud du graphe
for actor in G.nodes() :
    mbti_type = mbti_dict.get(actor, None) # None si pas de data
    G.nodes[actor]['mbti'] = mbti_type

# Vérifier
actors_with_mbti = [(n, d['mbti']) for n, d in G.nodes(data=True) if d['mbti'] is not None]
print(f"\nActeurs du graphe avec MBTI : {len(actors_with_mbti)}")
print("\nExemples :")
for name, mbti_type in actors_with_mbti[:10] :
    print(f"  {name:<30} : {mbti_type}")


Acteurs du graphe avec MBTI : 2401

Exemples :
  Sam Worthington                : ISFP
  Sigourney Weaver               : INTP
  Stephen Lang                   : ISTJ
  Michelle Rodriguez             : ESTP
  Giovanni Ribisi                : INTJ
  Laz Alonso                     : ESTJ
  Johnny Depp                    : INFP
  Orlando Bloom                  : ISFP
  Keira Knightley                : ENFP
  Stellan Skarsgård              : INTJ
